In [15]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

In [2]:
data = pd.read_csv('MFCCoutput.csv')
data = data.drop(data.columns[0], axis = 1)
print(data.head())

           0           1          2          3          4          5  \
0 -398.97055   92.846680  -6.212745  19.836878  -3.015201   9.478265   
1 -232.39255  115.044525 -21.028315  39.190132 -17.016842   7.619745   
2 -466.48450   89.272385  -8.458461  30.776363 -11.168960  18.305796   
3 -466.73505   62.805060  12.439709  29.304922  12.614448   9.676723   
4 -426.44970   80.985410  -4.792509  36.350883  -0.092283  19.156744   

          6          7         8         9  ...       119       120       121  \
0  5.134083   6.716100  1.437088  0.016323  ...  0.018726 -0.244671 -0.650705   
1 -2.724971   4.140653 -1.230497 -0.977595  ...  0.197640  0.564010  0.166091   
2  0.989266  10.417193  0.574027  1.563966  ...  0.232031  0.074036  0.237405   
3  1.418015  13.074185  0.037665  3.573357  ...  0.408432 -0.018611 -0.274066   
4 -5.060070  10.123955  2.196594  1.254182  ... -0.064134  0.120973  0.070115   

        122       123       124       125       126       127  class  
0 -0.2622

In [4]:
importance_df = pd.read_csv('feature_importance_basic.csv')
print(importance_df.head())
feature_names = importance_df["Feature"].tolist()
top_features = feature_names[:40]
print(top_features)

   Feature  CB_Importance  LGBM_Importance  ET_Importance  Avg_Importance
0        0       1.000000         1.000000       1.000000        1.000000
1       36       0.642909         0.740157       0.766617        0.716561
2       82       0.484306         0.732283       0.782581        0.666390
3       49       0.290911         0.787402       0.672607        0.583640
4       12       0.293238         0.464567       0.949464        0.569090
[0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 60, 93, 81, 38, 7, 62, 29, 61, 56]


In [7]:
selected_features = feature_names[:40]
print(selected_features)

drop_columns = []

for i in range(0,128):
    if i not in selected_features:
        drop_columns.append(i)

drop_columns = list(map(str, drop_columns))    

df_top_k = data.drop(columns=drop_columns, axis = 1)
print(df_top_k.head())

[0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 60, 93, 81, 38, 7, 62, 29, 61, 56]
           0          3          7        12        14         16        21  \
0 -398.97055  19.836878   6.716100 -2.605149  5.158936  10.482118  3.602238   
1 -232.39255  39.190132   4.140653 -1.054728  8.884685  -2.847462 -2.502778   
2 -466.48450  30.776363  10.417193  5.793727  3.335157   4.428913  6.333602   
3 -466.73505  29.304922  13.074185  6.860754 -1.749699   3.039867  4.576977   
4 -426.44970  36.350883  10.123955  3.329647  2.997393   2.873780  4.610694   

         23        24        26  ...        81        82        83        85  \
0  3.561909  0.169234  2.600260  ...  0.077412 -0.082618  0.150909  0.324120   
1 -0.466165 -3.284457 -0.991126  ... -0.183131  0.518272 -0.810806 -0.536778   
2  1.860267  0.856521  2.490161  ...  0.332677  0.154127  0.026485  0.547780   
3  1.654256  1.779491  1.162053  ...  0.9372

In [9]:
X = df_top_k.drop("class", axis=1)
y = df_top_k["class"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
cb = CatBoostClassifier()  
lgbm = LGBMClassifier()
et = ExtraTreesClassifier()

In [16]:
base_models = [
    ("CB", cb),
    ("LGBM", lgbm),
    ("ET", et)
]

meta_model = LogisticRegression()


ensemble_model = StackingClassifier(
    estimators=base_models, 
    final_estimator=meta_model, 
)


models = [
    ("CB", cb, "Reds"),
    ("LGBM", lgbm, "Blues"),
    ("ET", et, "Purples"),
    ("MVC", VotingClassifier(estimators=[
        ("cb", cb),
        ('lgbm', lgbm),
        ('et', et),
    ], voting='hard'), "Greens"),
    ("Ensemble", ensemble_model, "Oranges")
]

In [17]:
results = {}


for name, model, color in models:
    print(f"Training {name}...")
    model.fit(X_train, y_train) 
    y_pred = model.predict(X_test)  
    
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy  
    
    print(f"{name} Accuracy: {accuracy}\n")


print("Model Performance:")
for name, accuracy in results.items():
    print(f"{name}: {accuracy * 100:.2f}%")


Training CB...
Learning rate set to 0.017484
0:	learn: 0.6715489	total: 20.2ms	remaining: 20.2s
1:	learn: 0.6470611	total: 36.4ms	remaining: 18.2s
2:	learn: 0.6246864	total: 55.6ms	remaining: 18.5s
3:	learn: 0.6037155	total: 74.8ms	remaining: 18.6s
4:	learn: 0.5827888	total: 95.5ms	remaining: 19s
5:	learn: 0.5632686	total: 113ms	remaining: 18.7s
6:	learn: 0.5448074	total: 128ms	remaining: 18.2s
7:	learn: 0.5302636	total: 145ms	remaining: 18s
8:	learn: 0.5121413	total: 163ms	remaining: 17.9s
9:	learn: 0.4908362	total: 250ms	remaining: 24.7s
10:	learn: 0.4766773	total: 339ms	remaining: 30.5s
11:	learn: 0.4628845	total: 392ms	remaining: 32.3s
12:	learn: 0.4486524	total: 403ms	remaining: 30.6s
13:	learn: 0.4345928	total: 413ms	remaining: 29.1s
14:	learn: 0.4215515	total: 425ms	remaining: 27.9s
15:	learn: 0.4107617	total: 436ms	remaining: 26.8s
16:	learn: 0.3983151	total: 448ms	remaining: 25.9s
17:	learn: 0.3842384	total: 460ms	remaining: 25.1s
18:	learn: 0.3742534	total: 472ms	remaining: 2